# 03 — Validación Silver → Gold

Valida la jerarquía corregida y la granularidad de proceso sensible a la configuración.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('Raíz del proyecto:', PROJECT_ROOT)

In [ ]:
from pipelines.run_gold_pipeline import run_gold_pipeline
from transformation.silver_to_gold import read_silver
from schemas.gold_schema import GOLD_SCHEMA_VERSION, PROCESS_IDENTITY_COLUMNS

SILVER_ROOT = PROJECT_ROOT/'data'/'silver'
GOLD_ROOT = PROJECT_ROOT/'data'/'gold'
result = run_gold_pipeline(SILVER_ROOT, GOLD_ROOT, mode='incremental')
print('Schema Gold:', GOLD_SCHEMA_VERSION)
print('Estado:', result.get('status','SUCCESS'))
print('Identidad de proceso:', PROCESS_IDENTITY_COLUMNS)

In [ ]:
silver = read_silver(SILVER_ROOT)
dim_process = pd.read_parquet(GOLD_ROOT/'dim_process.parquet')
dim_date = pd.read_parquet(GOLD_ROOT/'dim_date.parquet')
facts = sorted((GOLD_ROOT/'fact_process_daily').glob('**/*.parquet'))
fact_daily = pd.concat([pd.read_parquet(p) for p in facts], ignore_index=True)
print('Silver:', len(silver))
print('dim_process:', len(dim_process))
print('dim_date:', len(dim_date))
print('Filas fact:', len(fact_daily))

## Granularidad y jerarquía

In [ ]:
assert not dim_process['process_key'].duplicated().any()
assert not fact_daily[['date_key','process_key']].duplicated().any()
required = {
    'factory_name','line_name','work_area_name','tightening_unit_id','tightening_unit_name',
    'process_step_id','process_step_name','substep_id','substep_number','substep_type'
}
assert required.issubset(dim_process.columns)
assert int(fact_daily['event_count'].sum()) == len(silver)
print('Granularidad Gold y reconciliación con Silver superadas.')
display(dim_process[[
    'factory_name','line_name','work_area_name','tightening_unit_name',
    'process_step_name','substep_number','substep_type','target_torque'
]].head(20))

## Reconciliaciones de KPIs

In [ ]:
assert (fact_daily['torque_applicable_count'] + fact_daily['torque_not_applicable_count']).eq(fact_daily['event_count']).all()
assert (fact_daily['torque_in_spec_count'] + fact_daily['torque_out_of_spec_count']).eq(fact_daily['torque_applicable_count']).all()
summary = pd.DataFrame({
    'metric':['events','torque_applicable','torque_in_spec','torque_out_of_spec'],
    'value':[
        int(fact_daily['event_count'].sum()), int(fact_daily['torque_applicable_count'].sum()),
        int(fact_daily['torque_in_spec_count'].sum()), int(fact_daily['torque_out_of_spec_count'].sum())
    ]
})
display(summary)

## Vista previa del serving para BI

In [ ]:
preview = fact_daily.merge(dim_process, on='process_key', how='left').merge(dim_date[['date_key','local_date']], on='date_key', how='left')
display(preview[[
    'local_date','factory_name','line_name','tightening_unit_name','process_step_name',
    'substep_number','event_count','torque_applicable_rate_pct','torque_in_spec_rate_pct','torque_out_of_spec_rate_pct'
]].head(20))